# Regional Feature Engineering

This notebook:

1. loads processed weekly storage (`01a`) and weekly weather (`01b`) for an EIA region;
2. joins storage and weather on week-ending Friday;
3. engineers model-ready features (calendar, weather lags, storage lags);
4. validates and exports the feature table for modeling notebooks.

Change `REGION` to rebuild features for Lower 48 or any of the five EIA storage regions.

For a routine refresh without re-running cells here, use:

```bash
gas-data refresh --region R48 --only features
```

In [ ]:
from pathlib import Path

import pandas as pd

from gas_forecast.data.export import save_versioned_parquet
from gas_forecast.data.features import (
    DEFAULT_WEATHER_MODEL_FEATURES,
    TARGET_COLUMN,
    build_weekly_model_features,
    validate_weekly_model_features,
)
from gas_forecast.data.paths import latest_processed_path
from gas_forecast.data.regions import region_slug, supported_storage_regions

In [ ]:
REGION = "R48"  # R48, R01, R02, R03, R04, R05
REGION_SLUG = region_slug(REGION)
PROCESSED_DIR = Path("../datasets/processed")

STORAGE_PATH = latest_processed_path(REGION, "weekly_storage", PROCESSED_DIR)
WEEKLY_WEATHER_PATH = latest_processed_path(REGION, "weekly_weather", PROCESSED_DIR)
FEATURES_PATH = latest_processed_path(REGION, "weekly_model_features", PROCESSED_DIR)

In [ ]:
storage = pd.read_parquet(STORAGE_PATH)
weekly_weather = pd.read_parquet(WEEKLY_WEATHER_PATH)

assert (storage["duoarea"] == REGION).all(), (
    f"Storage duoarea mismatch for {REGION}"
)
assert (weekly_weather["duoarea"] == REGION).all(), (
    f"Weather duoarea mismatch for {REGION}"
)

print(f"Storage rows: {len(storage):,}")
print(f"Weather rows: {len(weekly_weather):,}")

In [ ]:
weekly_model_features = build_weekly_model_features(
    storage,
    weekly_weather,
    region=REGION,
)

validate_weekly_model_features(weekly_model_features, region=REGION)

print(f"Target: {TARGET_COLUMN}")
print(f"Default weather model features: {DEFAULT_WEATHER_MODEL_FEATURES}")
weekly_model_features.head()

In [ ]:
save_versioned_parquet(
    weekly_model_features,
    PROCESSED_DIR,
    f"{REGION_SLUG}_weekly_model_features",
    save_latest=True,
)

## Optional: build all regions

Uncomment and run after `01a` and `01b` have been executed for each region.

In [ ]:
# from gas_forecast.pipelines import run_data_pipeline
#
# for region in supported_storage_regions():
#     run_data_pipeline(region, stages=("features",))